# Metastatic Cancer Detection — v2: stronger models + stain augmentation
**Deep Learning PBL · Saif Khattab (202312556) · Al-Zaytoonah University of Jordan**

Goal of v2: improve generalisation to **unseen slides** (Kaggle hidden test). v1 reached internal-test AUC 0.997 but
Kaggle private AUC 0.968 → the model does not transfer well across slides/labs.

| Change vs v1 | Why |
|---|---|
| **HED stain augmentation** (random H/E stain strength per image) | v1's gap points to stain/scanner shift between slides |
| Input upscaled to **192 px** for ImageNet backbones | pretrained features work better near their training resolution |
| Two new backbones: **EfficientNetB2** and **DenseNet121** | stronger + more diverse models for the ensemble |
| Up to **35 epochs**, EarlyStopping patience 6, `min_delta=0` (any improvement counts), LR halved after 2 flat epochs | train long; keep even small gains; always restore the best epoch |
| **Time budget** guard + per-model checkpoint & submission | Kaggle kills runs at 12 h; partial results are never lost |
| **Rank-average** ensemble | AUC depends only on ranking; rank averaging is robust to calibration differences |

Same split as v1 (seed 42) so results are directly comparable and can be ensembled with v1 offline.

> **Run as "Save & Run All (Commit)"** with GPU T4 x2 and Internet on.

## 1. Setup

In [ ]:
import os, glob, time, random, json, gc, traceback
RUN_START = time.time()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import rankdata

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------- Configuration ----------------
SEED          = 42
IMG_SIZE      = 96
TL_SIZE       = 192          # backbone input size (upscaled inside the model)
BATCH_SIZE    = 128
HEAD_EPOCHS   = 1            # warm-up of the new head with frozen backbone
MAX_EPOCHS    = 35           # upper bound for fine-tuning; EarlyStopping decides
FT_LR         = 1e-4
HEAD_LR       = 1e-3
ES_PATIENCE   = 6            # epochs without ANY improvement before stopping
ES_MIN_DELTA  = 0.0          # any improvement of val AUC counts
LR_PATIENCE   = 2            # halve LR after 2 flat epochs
MIN_LR        = 1e-7
# HED stain augmentation strength (per image, per stain channel)
HED_ALPHA     = 0.08         # stain scale  ~ U(1-a, 1+a)
HED_BETA      = 0.02         # stain shift  ~ U(-b, b)
# Time guard: Kaggle stops runs at 12 h; keep a safety margin for prediction/saving
TOTAL_BUDGET_H   = 11.0
RESERVE_AFTER_H  = 0.5       # time reserved after each model for TTA prediction + saving
MODELS_TO_TRAIN  = ['EfficientNetB2', 'DenseNet121']
TARGET_RECALL = 0.95
OUT_DIR       = '/kaggle/working'

keras.mixed_precision.set_global_policy('mixed_float16')
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
set_seed(SEED)

def hours_left():
    return TOTAL_BUDGET_H - (time.time() - RUN_START) / 3600

print('TensorFlow:', tf.__version__, '| Keras:', keras.__version__)
print('GPUs      :', tf.config.list_physical_devices('GPU'))

## 2. Data and the same 70/15/15 split as v1

In [ ]:
candidates = ['/kaggle/input/histopathologic-cancer-detection',
              '/kaggle/input/competitions/histopathologic-cancer-detection']
DATA_DIR = next((d for d in candidates if os.path.exists(os.path.join(d, 'train_labels.csv'))), None)
if DATA_DIR is None:
    found = glob.glob('/kaggle/input/*/train_labels.csv') + glob.glob('/kaggle/input/*/*/train_labels.csv')
    assert found, 'train_labels.csv not found - check that the competition data is attached.'
    DATA_DIR = os.path.dirname(found[0])
TRAIN_DIR, TEST_DIR = os.path.join(DATA_DIR, 'train'), os.path.join(DATA_DIR, 'test')

df = pd.read_csv(os.path.join(DATA_DIR, 'train_labels.csv')).reset_index(drop=True)
kaggle_test_ids = sorted(f[:-4] for f in os.listdir(TEST_DIR) if f.endswith('.tif'))

# identical to v1 (SUBSET_SIZE=None -> df.copy().reset_index)
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
val_df,  test_df  = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)
print({k: len(v) for k, v in [('train', train_df), ('val', val_df), ('test', test_df), ('kaggle', kaggle_test_ids)]})
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d.to_csv(f'{OUT_DIR}/split_{name}.csv', index=False)

In [ ]:
def load_images(ids, folder, workers=32):
    t0 = time.time()
    out = np.empty((len(ids), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    paths = [os.path.join(folder, f'{i}.tif') for i in ids]
    def work(k):
        with Image.open(paths[k]) as im:
            out[k] = np.asarray(im.convert('RGB'), dtype=np.uint8)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(ex.map(work, range(len(paths))))
    print(f'  loaded {len(paths):,} images in {time.time() - t0:.0f}s')
    return out

X_train = load_images(train_df['id'].values, TRAIN_DIR)
X_val   = load_images(val_df['id'].values,   TRAIN_DIR)
X_test  = load_images(test_df['id'].values,  TRAIN_DIR)
X_kag   = load_images(kaggle_test_ids,       TEST_DIR)
y_train = train_df['label'].values.astype('float32')
y_val   = val_df['label'].values.astype('float32')
y_test  = test_df['label'].values.astype('float32')
print(f'Hours left in budget: {hours_left():.2f}')

## 3. Pipeline with HED stain augmentation
RGB → optical density → H/E/D stain space (Ruifrok & Johnston matrix, as in `skimage.color.rgb2hed`);
each stain is randomly scaled and shifted per image, then converted back to RGB. Plus the 8 dihedral transforms
(keep the central 32×32 region in place) and mild brightness/contrast. **No cropping.**

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
RGB_FROM_HED = tf.constant([[0.65, 0.70, 0.29], [0.07, 0.99, 0.11], [0.27, 0.57, 0.78]], tf.float32)
HED_FROM_RGB = tf.linalg.inv(RGB_FROM_HED)
LOG_EPS = float(np.log(1e-6))

def hed_jitter(x):
    # x: float32 batch in [0,1], shape (b,h,w,3)
    b = tf.shape(x)[0]
    s = tf.einsum('bhwc,cd->bhwd', tf.math.log(tf.maximum(x, 1e-6)) / LOG_EPS, HED_FROM_RGB)
    alpha = tf.random.uniform([b, 1, 1, 3], 1 - HED_ALPHA, 1 + HED_ALPHA)
    beta  = tf.random.uniform([b, 1, 1, 3], -HED_BETA, HED_BETA)
    s = s * alpha + beta
    rgb = tf.exp(tf.einsum('bhwd,dc->bhwc', s * LOG_EPS, RGB_FROM_HED))
    return tf.clip_by_value(rgb, 0.0, 1.0)

def augment_batch(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    b = tf.shape(x)[0]
    x = tf.where(tf.random.uniform([b, 1, 1, 1]) < 0.5, tf.reverse(x, [2]), x)
    x = tf.where(tf.random.uniform([b, 1, 1, 1]) < 0.5, tf.reverse(x, [1]), x)
    x = tf.where(tf.random.uniform([b, 1, 1, 1]) < 0.5, tf.transpose(x, [0, 2, 1, 3]), x)
    x = hed_jitter(x)
    x = x + tf.random.uniform([b, 1, 1, 1], -0.05, 0.05)
    mean = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)
    x = (x - mean) * tf.random.uniform([b, 1, 1, 1], 0.9, 1.1) + mean
    return tf.clip_by_value(x, 0.0, 1.0) * 255.0, y

def to_float(x, y):
    return tf.cast(x, tf.float32), y

def make_ds(X, y=None, training=False, batch=BATCH_SIZE):
    if y is None:
        y = np.zeros(len(X), dtype='float32')
    idx = tf.data.Dataset.range(len(X))
    if training:
        idx = idx.shuffle(len(X), seed=SEED, reshuffle_each_iteration=True)
    idx = idx.batch(batch)
    def gather(i):
        xb, yb = tf.numpy_function(lambda j: (X[j], y[j]), [i], [tf.uint8, tf.float32])
        xb.set_shape([None, IMG_SIZE, IMG_SIZE, 3]); yb.set_shape([None])
        return xb, yb
    ds = idx.map(gather, num_parallel_calls=AUTOTUNE)
    ds = ds.map(augment_batch if training else to_float, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds, val_ds = make_ds(X_train, y_train, training=True), make_ds(X_val, y_val)

# sanity check of the HED round trip (no jitter) and a visual check
x0 = tf.constant(X_train[:4].astype('float32') / 255.0)
s0 = tf.einsum('bhwc,cd->bhwd', tf.math.log(tf.maximum(x0, 1e-6)) / LOG_EPS, HED_FROM_RGB)
rt = tf.exp(tf.einsum('bhwd,dc->bhwc', s0 * LOG_EPS, RGB_FROM_HED))
print('HED round-trip max error:', float(tf.reduce_max(tf.abs(rt - x0))))
aug, _ = augment_batch(tf.constant(np.repeat(X_train[:1], 8, axis=0)), tf.zeros(8))
fig, axes = plt.subplots(1, 9, figsize=(18, 2.4))
axes[0].imshow(X_train[0]); axes[0].set_title('original')
for ax, im in zip(axes[1:], aug.numpy().astype('uint8')):
    ax.imshow(im)
for ax in axes:
    ax.add_patch(Rectangle((31.5, 31.5), 32, 32, fill=False, edgecolor='lime', lw=1.2)); ax.axis('off')
plt.suptitle('HED stain augmentation examples'); plt.tight_layout(); plt.show()

## 4. Models, callbacks, evaluation helpers

In [ ]:
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train.astype(int))
CLASS_WEIGHT = {0: float(cw[0]), 1: float(cw[1])}

def get_metrics():
    return [keras.metrics.AUC(name='auc'), keras.metrics.Recall(name='recall'),
            keras.metrics.Precision(name='precision'), keras.metrics.BinaryAccuracy(name='accuracy')]

class TimeBudget(keras.callbacks.Callback):
    # Stops training cleanly if the next epoch would not finish before `deadline` (epoch time estimated online).
    def __init__(self, deadline):
        super().__init__(); self.deadline = deadline; self.t = None; self.durations = []
    def on_epoch_begin(self, epoch, logs=None):
        self.t = time.time()
    def on_epoch_end(self, epoch, logs=None):
        self.durations.append(time.time() - self.t)
        if time.time() + max(self.durations) > self.deadline:
            print(f'\n[TimeBudget] stopping after epoch {epoch + 1} to respect the run time limit.')
            self.model.stop_training = True

def get_callbacks(weights_path, deadline, patience=ES_PATIENCE):
    return [keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=patience,
                                          min_delta=ES_MIN_DELTA, restore_best_weights=True, verbose=1),
            keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=LR_PATIENCE,
                                              min_delta=ES_MIN_DELTA, min_lr=MIN_LR, verbose=1),
            keras.callbacks.ModelCheckpoint(weights_path, monitor='val_auc', mode='max',
                                            save_best_only=True, save_weights_only=True),
            TimeBudget(deadline)]

def build_model(name):
    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Resizing(TL_SIZE, TL_SIZE, interpolation='bilinear', name='upscale')(inp)
    if name == 'EfficientNetB2':          # includes its own preprocessing, expects [0,255]
        base = keras.applications.EfficientNetB2(include_top=False, weights='imagenet', input_tensor=x)
        cam_layer = 'top_activation'
    elif name == 'DenseNet121':           # expects ImageNet mean/std normalised input
        x = layers.Rescaling(1.0 / 255)(x)
        x = layers.Normalization(mean=[0.485, 0.456, 0.406], variance=[0.229 ** 2, 0.224 ** 2, 0.225 ** 2])(x)
        base = keras.applications.DenseNet121(include_top=False, weights='imagenet', input_tensor=x)
        cam_layer = 'relu'
    else:
        raise ValueError(name)
    y = layers.GlobalAveragePooling2D(name='gap')(base.output)
    y = layers.Dropout(0.3)(y)
    out = layers.Dense(1, activation='sigmoid', dtype='float32', name='output')(y)
    return keras.Model(inp, out, name=name), base, cam_layer

def set_backbone_trainable(base, trainable):
    for l in base.layers:
        l.trainable = trainable and not isinstance(l, layers.BatchNormalization)

def predict(model, X):
    base = make_ds(X, batch=256)
    preds = []
    for t in range(8):
        def tf_t(x, y, t=t):
            if t & 1: x = tf.reverse(x, [2])
            if t & 2: x = tf.reverse(x, [1])
            if t & 4: x = tf.transpose(x, [0, 2, 1, 3])
            return x
        preds.append(model.predict(base.map(tf_t), verbose=0).ravel())
    return np.mean(preds, axis=0)

def evaluate(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    prec = tp / (tp + fp) if tp + fp else 0.0
    return {'threshold': round(float(thr), 4), 'AUC': roc_auc_score(y, p), 'Accuracy': (tp + tn) / len(y),
            'Sensitivity (Recall)': sens, 'Specificity': spec, 'Precision': prec,
            'F1': 2 * prec * sens / (prec + sens) if prec + sens else 0.0,
            'FN (missed tumours)': int(fn), 'FP (false alarms)': int(fp)}

def rank_avg(preds):
    return np.mean([rankdata(p) / len(p) for p in preds], axis=0)

## 5. Preflight (≈3 min): every step on a few batches so errors appear now, not hours later

In [ ]:
t0 = time.time()
small = make_ds(X_train[:BATCH_SIZE * 2], y_train[:BATCH_SIZE * 2], training=True)
for name in MODELS_TO_TRAIN:
    m, b, cam_layer = build_model(name)
    set_backbone_trainable(b, False)
    m.compile(optimizer=keras.optimizers.Adam(HEAD_LR), loss='binary_crossentropy', metrics=get_metrics())
    m.fit(small, epochs=1, steps_per_epoch=1, verbose=0)
    set_backbone_trainable(b, True)
    m.compile(optimizer=keras.optimizers.Adam(FT_LR), loss='binary_crossentropy', metrics=get_metrics())
    m.fit(small, validation_data=make_ds(X_val[:256], y_val[:256]), epochs=1, steps_per_epoch=2,
          class_weight=CLASS_WEIGHT, callbacks=get_callbacks('/tmp/pf.weights.h5', time.time() + 3600), verbose=0)
    m.load_weights('/tmp/pf.weights.h5')
    p = predict(m, X_val[:64]); assert p.shape == (64,) and np.isfinite(p).all()
    _ = m.get_layer(cam_layer)
    print(f'  {name}: OK (full batch {BATCH_SIZE} at {TL_SIZE}px fits in GPU memory)')
    del m, b; keras.backend.clear_session(); gc.collect()
pd.DataFrame({'id': kaggle_test_ids[:64], 'label': rank_avg([np.random.rand(64), np.random.rand(64)])}).to_csv('/tmp/s.csv', index=False)
set_seed(SEED)
print(f'Preflight passed in {time.time() - t0:.0f}s | hours left: {hours_left():.2f}')

## 6. Train each model (results are saved right after each model, so nothing is lost)

In [ ]:
results, P_VAL, P_TEST, P_KAG, HIST = {}, {}, {}, {}, {}
for i, name in enumerate(MODELS_TO_TRAIN):
    try:
        models_left = len(MODELS_TO_TRAIN) - i
        budget_h = (hours_left() - RESERVE_AFTER_H * models_left) / models_left
        if budget_h < 0.5:
            print(f'Skipping {name}: not enough time left ({hours_left():.2f} h)'); continue
        deadline = time.time() + budget_h * 3600
        print(f'\n===== {name} | training budget {budget_h:.2f} h =====')
        set_seed(SEED)
        model, base, cam_layer = build_model(name)
        W = f'{OUT_DIR}/{name}.weights.h5'

        set_backbone_trainable(base, False)          # stage 1: head warm-up
        model.compile(optimizer=keras.optimizers.Adam(HEAD_LR), loss='binary_crossentropy', metrics=get_metrics())
        h1 = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=CLASS_WEIGHT, verbose=2)

        set_backbone_trainable(base, True)           # stage 2: full fine-tuning (BatchNorm frozen)
        model.compile(optimizer=keras.optimizers.Adam(FT_LR), loss='binary_crossentropy', metrics=get_metrics())
        t0 = time.time()
        h2 = model.fit(train_ds, validation_data=val_ds, epochs=MAX_EPOCHS, class_weight=CLASS_WEIGHT,
                       callbacks=get_callbacks(W, deadline), verbose=2)
        model.load_weights(W)
        va = h2.history['val_auc']
        HIST[name] = h2.history
        print(f'{name}: {len(va)} fine-tune epochs, best epoch {int(np.argmax(va)) + 1}, '
              f'best val AUC {max(va):.4f}, {(time.time() - t0) / 3600:.2f} h')

        P_VAL[name], P_TEST[name], P_KAG[name] = predict(model, X_val), predict(model, X_test), predict(model, X_kag)
        results[name] = {'val': evaluate(y_val, P_VAL[name], 0.5), 'test': evaluate(y_test, P_TEST[name], 0.5),
                         'epochs': len(va), 'best_epoch': int(np.argmax(va)) + 1}
        tag = name.lower()
        pd.DataFrame({'id': kaggle_test_ids, 'label': P_KAG[name]}).to_csv(f'{OUT_DIR}/submission_{tag}.csv', index=False)
        pd.DataFrame({'id': val_df['id'].values, 'label': y_val.astype(int), 'p': P_VAL[name]}).to_csv(f'{OUT_DIR}/val_pred_{tag}.csv', index=False)
        pd.DataFrame({'id': test_df['id'].values, 'label': y_test.astype(int), 'p': P_TEST[name]}).to_csv(f'{OUT_DIR}/test_pred_{tag}.csv', index=False)
        model.save(f'{OUT_DIR}/{name}.keras')
        print(f'{name} saved | val AUC (TTA) {results[name]["val"]["AUC"]:.4f} | test AUC {results[name]["test"]["AUC"]:.4f} '
              f'| hours left {hours_left():.2f}')
        fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
        for ax, m_ in zip(axes[:2], ['loss', 'auc']):
            ax.plot(h2.history[m_], label='train'); ax.plot(h2.history[f'val_{m_}'], label='val'); ax.legend(); ax.set_title(f'{name} {m_}')
        axes[2].plot(h2.history.get('learning_rate', [])); axes[2].set_yscale('log'); axes[2].set_title('learning rate')
        plt.tight_layout(); plt.show()
        del model, base; keras.backend.clear_session(); gc.collect()
    except Exception:
        print(f'!!! {name} failed - continuing with the other models'); traceback.print_exc()
        keras.backend.clear_session(); gc.collect()

## 7. Ensemble of the new models (rank average) and summary

In [ ]:
names = list(P_VAL)
assert names, 'no model finished'
if len(names) > 1:
    P_VAL['Ensemble v2'] = rank_avg([P_VAL[n] for n in names])
    P_TEST['Ensemble v2'] = rank_avg([P_TEST[n] for n in names])
    P_KAG['Ensemble v2'] = rank_avg([P_KAG[n] for n in names])
    pd.DataFrame({'id': kaggle_test_ids, 'label': P_KAG['Ensemble v2']}).to_csv(f'{OUT_DIR}/submission_ensemble_v2.csv', index=False)
summary = pd.DataFrame({n: {'val AUC': roc_auc_score(y_val, P_VAL[n]), 'test AUC': roc_auc_score(y_test, P_TEST[n])}
                        for n in P_VAL}).T.sort_values('val AUC', ascending=False)
display(summary.round(5))
summary.to_csv(f'{OUT_DIR}/summary_v2.csv')
with open(f'{OUT_DIR}/results_v2.json', 'w') as f:
    json.dump({'results': results, 'config': {'TL_SIZE': TL_SIZE, 'MAX_EPOCHS': MAX_EPOCHS, 'ES_PATIENCE': ES_PATIENCE,
               'HED_ALPHA': HED_ALPHA, 'HED_BETA': HED_BETA, 'FT_LR': FT_LR}}, f, indent=2, default=float)
print('Best on validation:', summary.index[0])
print(f'Total run time: {(time.time() - RUN_START) / 3600:.2f} h')
print(sorted(os.listdir(OUT_DIR)))